# Chilled-water loop: replacement pump selection
Duty point: 50 L/s at 28 m. System curve: 12 m static head plus friction proportional to flow squared.
Criteria: operating flow within 70-110% of best-efficiency flow, NPSH margin of at least 1.5 m (NPSH available 6.0 m).

## 1. Load the data and check units

In [1]:
import numpy as np
import pandas as pd
from pump_helpers import fit_all, fit_report, plot_operating_points

df = pd.read_csv("pump_curves.csv")
df.head(10)

,pump,flow_Ls,head_m,efficiency_pct,npshr_m
0,A,10.0,41.0,39.3,2.4
1,A,20.0,39.3,57.4,2.5
2,A,30.0,36.6,69.7,2.8
3,A,40.0,32.8,76.3,3.2
4,A,50.0,27.9,77.1,3.7
5,A,60.0,21.9,72.3,4.3
6,A,70.0,14.8,61.7,5.0
7,A,80.0,6.6,45.3,5.8
8,B,10.0,39.5,38.8,2.6
9,B,20.0,38.1,57.0,2.7


In [2]:
df.groupby("pump").agg(["min", "max"])

flow_Ls       head_m       efficiency_pct       npshr_m     
         min   max    min   max            min   max     min  max
pump                                                             
A       10.0  80.0    6.6  41.0           39.3  77.1     2.4  5.8
B       10.0  80.0    9.3  39.5           38.8  83.6     2.6  6.4
C       10.0  80.0   11.5  38.6           38.5  87.9     2.7  8.5

## 2. Fit a quadratic curve for each pump

In [3]:
fits = fit_all(df)
fit_report(df, fits)

,pump,head_m,efficiency_pct,npshr_m
0,A,0.038,0.053,0.038
1,B,0.023,0.043,0.040
2,C,0.015,0.045,0.032


## 3. System curve and operating points

In [4]:
# system curve H = static_head + k * Q^2, calibrated so that 50 L/s needs 28 m
static_head = 12.0
k = (28 - static_head) / 50**2
system = np.poly1d([k, 0, static_head])
k

0.0064

In [5]:
op = {}
for p, f in fits.items():
    roots = (f["head"] - system).roots
    q = roots[(roots.imag == 0) & (roots.real > 0)].real[0]
    op[p] = q
op

{'A': np.float64(49.905006391233314),
 'B': np.float64(49.999468393326474),
 'C': np.float64(50.19878324334869)}

### Hand check on one pump

In [6]:
p = "B"
q = op[p]
fits[p]["head"](q) - system(q)

np.float64(0.0)

## 4. Criteria and annual energy cost

In [7]:
rho, g, hours, tariff, npsh_avail = 1000, 9.81, 6000, 0.12, 6.0

In [8]:
rows = []
for p, f in fits.items():
    q = op[p]
    h = f["head"](q)
    eta = f["eff"](q) / 100
    q_bep = -f["eff"].coeffs[1] / (2 * f["eff"].coeffs[0])
    margin = npsh_avail - f["npshr"](q)
    kw = rho * g * (q / 1000) * h / eta / 1000
    cost = kw * hours * tariff
    rows.append(dict(pump=p, flow_Ls=q, head_m=h, eff_pct=eta * 100, pct_of_bep=100 * q / q_bep,
                     npsh_margin_m=margin, shaft_kw=kw, annual_cost=cost))
res = pd.DataFrame(rows).round(2)
res

,pump,flow_Ls,head_m,eff_pct,pct_of_bep,npsh_margin_m,shaft_kw,annual_cost
0,A,49.91,27.94,77.17,107.33,2.32,17.72,12761.51
1,B,50.00,28.00,83.63,92.61,1.99,16.42,11824.03
2,C,50.20,28.13,86.68,86.55,1.08,15.98,11505.38


In [9]:
res["pass"] = res.pct_of_bep.between(70, 110) & (res.npsh_margin_m >= 1.5)
res

,pump,flow_Ls,head_m,eff_pct,pct_of_bep,npsh_margin_m,shaft_kw,annual_cost,pass
0,A,49.91,27.94,77.17,107.33,2.32,17.72,12761.51,True
1,B,50.00,28.00,83.63,92.61,1.99,16.42,11824.03,True
2,C,50.20,28.13,86.68,86.55,1.08,15.98,11505.38,False


## 5. Save the table and the chart

In [10]:
res.to_csv("operating_points.csv", index=False)
plot_operating_points(df, fits, system, op, "operating_points.png", passed=res.set_index("pump")["pass"])

'operating_points.png'